# Physical Activity Analysis for Ages 18-24

This notebook analyses physical activity and inactivity patterns for people aged **18-24** using the ABS source table.

## Goals
- load the raw Excel table
- extract the **18-24** column
- clean and wrangle the physical-activity categories
- separate the data into meaningful metric groups
- check that each distribution is complete
- visualise the distributions clearly without mixing incompatible duration bands
- estimate an approximate average duration for each metric group

## Why this matters
This analysis helps show how time is distributed across inactivity, light activity, moderate activity, vigorous activity, and combined moderate-vigorous activity for young adults.


In [16]:
import pandas as pd
import plotly.express as px
import re
import plotly.graph_objects as go


## Step 1: Load the raw Excel data

The worksheet is formatted for reporting, not analysis.  
So the first step is to load it exactly as it appears and inspect the raw rows before cleaning anything.


In [17]:
file_path = r"C:\Users\heryn\OneDrive\Documents\Semester 1 2026\FIT 5120\iteration 1 brainboost\project brain health\Measured-physical-activity-and-sleep-all\MPASDC01.xlsx"

df = pd.read_excel(file_path, sheet_name="Table 1.1_Proportions", header=None)

print(df.head(20))


                                                   0   \
0   This tab contains table 1.1, proportion of per...   
1                     Australian Bureau of Statistics   
2   Table 1.1 Physical activity, by age and sex(a)...   
3   National Nutrition and Physical Activity Surve...   
4                                                 NaN   
5                                                 NaN   
6                                                 NaN   
7                            Average daily inactivity   
8                                   Less than 9 hours   
9                             9 to less than 10 hours   
10                           10 to less than 11 hours   
11                           11 to less than 12 hours   
12                           12 to less than 13 hours   
13                           13 to less than 14 hours   
14                           14 to less than 15 hours   
15                                   15 hours or more   
16                             

## Step 2: Extract the physical-activity table for ages 18-24

This step:
- uses the correct row as the header row
- keeps only the table body
- finds the age column for **18-24**
- renames the selected measure column to `value`


In [18]:
columns = ["category"] + df.iloc[5, 1:].tolist()

table = df.iloc[7:55, 0:len(columns)].copy()
table.columns = columns

table.columns = [str(col).strip() for col in table.columns]
table["category"] = table["category"].astype(str).str.strip()

age_col = next(col for col in table.columns if "18" in str(col) and "24" in str(col))

activity_18_24 = table[["category", age_col]].copy()
activity_18_24 = activity_18_24.rename(columns={age_col: "value"})

print(activity_18_24.head(25))


                                             category     value
7                            Average daily inactivity       NaN
8                                   Less than 9 hours       7.2
9                             9 to less than 10 hours        11
10                           10 to less than 11 hours      17.1
11                           11 to less than 12 hours      29.7
12                           12 to less than 13 hours      11.1
13                           13 to less than 14 hours      12.5
14                           14 to less than 15 hours       7.1
15                                   15 hours or more       4.4
16                                      Total persons       100
17            Average daily inactivity (Mean – h:min)  11:38:00
18              Average daily light physical activity       NaN
19                                  Less than 2 hours      28.3
20                             2 to less than 3 hours      43.7
21                             3 to less

## Step 3: Clean and wrangle the extracted data

The raw extract still contains:
- heading rows
- total rows
- footnotes
- note rows
- values stored as text

This step removes non-data rows and converts percentages into numeric values.


In [19]:
activity_18_24 = activity_18_24.dropna(subset=["category"]).copy()

activity_18_24 = activity_18_24[
    ~activity_18_24["category"].str.contains(
        "Footnotes|Commonwealth of Australia|Proportions may not add up",
        case=False,
        na=False,
        regex=True,
    )
].copy()

activity_18_24["value"] = (
    activity_18_24["value"]
    .astype(str)
    .str.replace("#", "", regex=False)
    .str.replace(",", ".", regex=False)
)

activity_18_24["value"] = pd.to_numeric(activity_18_24["value"], errors="coerce")

print(activity_18_24)


                                             category  value
7                            Average daily inactivity    NaN
8                                   Less than 9 hours    7.2
9                             9 to less than 10 hours   11.0
10                           10 to less than 11 hours   17.1
11                           11 to less than 12 hours   29.7
12                           12 to less than 13 hours   11.1
13                           13 to less than 14 hours   12.5
14                           14 to less than 15 hours    7.1
15                                   15 hours or more    4.4
16                                      Total persons  100.0
17            Average daily inactivity (Mean – h:min)    NaN
18              Average daily light physical activity    NaN
19                                  Less than 2 hours   28.3
20                             2 to less than 3 hours   43.7
21                             3 to less than 4 hours   19.9
22                      

## Step 4: Assign each row to a metric group

The ABS table is organised in blocks.  
Each heading starts a new metric group, and the rows underneath belong to that group until the next heading appears.

The metric groups used here are:
- `inactivity`
- `light_activity`
- `moderate_vigorous`
- `moderate`
- `vigorous`


In [20]:
group_headings = {
    "Average daily inactivity": "inactivity",
    "Average daily light physical activity": "light_activity",
    "Average daily moderate and vigorous physical activity": "moderate_vigorous",
    "Average daily moderate physical activity": "moderate",
    "Average daily vigorous physical activity": "vigorous",
}

current_group = None
groups = []

for category in activity_18_24["category"]:
    matched_group = None
    for heading, group_name in group_headings.items():
        if heading in category and "Mean" not in category:
            matched_group = group_name
            break

    if matched_group is not None:
        current_group = matched_group
        groups.append(None)
    else:
        groups.append(current_group)

activity_18_24["metric_group"] = groups

cleaned = activity_18_24[
    ~activity_18_24["category"].str.contains(
        "Average daily|Total persons|Footnotes|Commonwealth of Australia|Proportions may not add up",
        case=False,
        na=False,
        regex=True,
    )
].copy()

cleaned = cleaned.dropna(subset=["metric_group", "value"])
cleaned = cleaned[["metric_group", "category", "value"]]

metric_labels = {
    "inactivity": "Daily inactivity",
    "light_activity": "Light physical activity",
    "moderate_vigorous": "Moderate + vigorous activity",
    "moderate": "Moderate physical activity",
    "vigorous": "Vigorous physical activity",
}

cleaned["metric_label"] = cleaned["metric_group"].map(metric_labels)

print(cleaned)


         metric_group                        category  value  \
8          inactivity               Less than 9 hours    7.2   
9          inactivity         9 to less than 10 hours   11.0   
10         inactivity        10 to less than 11 hours   17.1   
11         inactivity        11 to less than 12 hours   29.7   
12         inactivity        12 to less than 13 hours   11.1   
13         inactivity        13 to less than 14 hours   12.5   
14         inactivity        14 to less than 15 hours    7.1   
15         inactivity                15 hours or more    4.4   
19     light_activity               Less than 2 hours   28.3   
20     light_activity          2 to less than 3 hours   43.7   
21     light_activity          3 to less than 4 hours   19.9   
22     light_activity                 4 hours or more    8.1   
26  moderate_vigorous            Less than 30 minutes    1.8   
27  moderate_vigorous  30 minutes to less than 1 hour    9.8   
28  moderate_vigorous        1 to less t

## Step 5: Check the cleaned distributions

Before plotting anything, it is important to confirm that the wrangled groups make sense.

This check looks at:
- how many duration bands each metric group contains
- whether the percentages sum to about **100%** within each group

If the totals are far from 100, it usually means rows were assigned incorrectly or some categories were lost during cleaning.


In [21]:
quality_check = (
    cleaned.groupby(["metric_group", "metric_label"], as_index=False)
    .agg(
        band_count=("category", "count"),
        percentage_total=("value", "sum"),
        min_value=("value", "min"),
        max_value=("value", "max"),
    )
)

quality_check["percentage_total"] = quality_check["percentage_total"].round(1)

print(quality_check)


        metric_group                  metric_label  band_count  \
0         inactivity              Daily inactivity           8   
1     light_activity       Light physical activity           4   
2           moderate    Moderate physical activity           7   
3  moderate_vigorous  Moderate + vigorous activity           7   
4           vigorous    Vigorous physical activity           4   

   percentage_total  min_value  max_value  
0             100.1        4.4       29.7  
1             100.0        8.1       43.7  
2             100.1        1.8       28.4  
3             100.0        1.8       25.9  
4             100.0        7.4       59.6  


## Step 6: Order the duration bands inside each metric group

Each metric uses its own duration bands, so the ordering must be handled **within** each group.

This is also why the notebook avoids a single grouped chart that mixes inactivity and physical-activity bands on the same x-axis.  
Those bands are not directly comparable and would create a misleading visual.


In [22]:
group_orders = {
    "inactivity": [
        "Less than 9 hours",
        "9 to less than 10 hours",
        "10 to less than 11 hours",
        "11 to less than 12 hours",
        "12 to less than 13 hours",
        "13 to less than 14 hours",
        "14 to less than 15 hours",
        "15 hours or more",
    ],
    "light_activity": [
        "Less than 2 hours",
        "2 to less than 3 hours",
        "3 to less than 4 hours",
        "4 hours or more",
    ],
    "moderate_vigorous": [
        "Less than 30 minutes",
        "30 minutes to less than 1 hour",
        "1 to less than 1.5 hours",
        "1.5 to less than 2 hours",
        "2 to less than 2.5 hours",
        "2.5 to less than 3 hours",
        "3 hours or more",
    ],
    "moderate": [
        "Less than 30 minutes",
        "30 minutes to less than 1 hour",
        "1 to less than 1.5 hours",
        "1.5 to less than 2 hours",
        "2 to less than 2.5 hours",
        "2.5 to less than 3 hours",
        "3 hours or more",
    ],
    "vigorous": [
        "0 minutes",
        "1?4 minutes",
        "5?9 minutes",
        "10 minutes or more",
    ],
}

order_lookup = {
    (group_name, label): order
    for group_name, labels in group_orders.items()
    for order, label in enumerate(labels)
}

cleaned["band_order"] = cleaned.apply(
    lambda row: order_lookup.get((row["metric_group"], row["category"]), 999),
    axis=1,
)

cleaned = cleaned.sort_values(["metric_group", "band_order"]).reset_index(drop=True)

print(cleaned)


         metric_group                        category  value  \
0          inactivity               Less than 9 hours    7.2   
1          inactivity         9 to less than 10 hours   11.0   
2          inactivity        10 to less than 11 hours   17.1   
3          inactivity        11 to less than 12 hours   29.7   
4          inactivity        12 to less than 13 hours   11.1   
5          inactivity        13 to less than 14 hours   12.5   
6          inactivity        14 to less than 15 hours    7.1   
7          inactivity                15 hours or more    4.4   
8      light_activity               Less than 2 hours   28.3   
9      light_activity          2 to less than 3 hours   43.7   
10     light_activity          3 to less than 4 hours   19.9   
11     light_activity                 4 hours or more    8.1   
12           moderate            Less than 30 minutes    1.8   
13           moderate  30 minutes to less than 1 hour   10.8   
14           moderate        1 to less t

## Step 7: Visualise each distribution separately

These charts keep each metric group separate so the duration bands stay interpretable.

This is a cleaner choice than combining everything into one grouped chart, because:
- inactivity uses **hours**
- some activity metrics use **hours and half-hours**
- vigorous activity uses **minutes**

Keeping them separate makes the analysis easier to read and more accurate.


In [23]:
def plot_duration_distribution(data, metric_group, title, x_label):
    subset = data[data["metric_group"] == metric_group].copy()
    subset = subset.sort_values("band_order")

    fig = px.bar(
        subset,
        x="value",
        y="category",
        orientation="h",
        text="value",
        title=title,
        labels={"value": "Percentage", "category": x_label},
    )

    fig.update_traces(
        texttemplate="%{text:.1f}%",
        textposition="outside",
        hovertemplate=(
            f"{x_label}: %{{y}}<br>"
            "Percentage: %{x:.1f}%<extra></extra>"
        ),
    )

    fig.update_layout(
        template="plotly_white",
        height=550,
        title_x=0.5,
        yaxis=dict(categoryorder="array", categoryarray=subset["category"][::-1]),
    )

    fig.show()


plot_duration_distribution(cleaned, "inactivity", "Daily Inactivity Distribution for Ages 18-24", "Inactivity duration")
plot_duration_distribution(cleaned, "light_activity", "Light Physical Activity Distribution for Ages 18-24", "Activity duration")
plot_duration_distribution(cleaned, "moderate_vigorous", "Moderate and Vigorous Activity for Ages 18-24", "Activity duration")
plot_duration_distribution(cleaned, "moderate", "Moderate Physical Activity Distribution for Ages 18-24", "Activity duration")
plot_duration_distribution(cleaned, "vigorous", "Vigorous Activity Distribution for Ages 18-24", "Activity duration")


In [24]:

main_cleaned = cleaned[
    cleaned["metric_group"].isin(["inactivity", "moderate_vigorous"])
].copy()

main_cleaned["metric_group"] = main_cleaned["metric_group"].replace({
    "inactivity": "Daily Inactivity",
    "moderate_vigorous": "Moderate + Vigorous Activity"
})

fig = px.bar(
    main_cleaned,
    x="category",
    y="value",
    color="metric_group",
    barmode="group",
    text="value",
    title="Movement Profile for Ages 18–24",
    labels={
        "category": "Duration band",
        "value": "Percentage",
        "metric_group": "Metric"
    },
    hover_data={"value": ":.1f"}
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=650,
    title_x=0.5,
    xaxis_tickangle=-30
)

fig.show()

In [25]:

# filter inactivity only
inactivity = cleaned[cleaned["metric_group"] == "inactivity"].copy()

# average inactivity for 18–24
avg_inactivity = 11.38

fig = go.Figure()

# bar chart
fig.add_trace(go.Bar(
    x=inactivity["category"],
    y=inactivity["value"],
    text=inactivity["value"],
    textposition="outside",
    name="Percentage of 18–24",
    hovertemplate="Band: %{x}<br>Percentage: %{y:.1f}%<extra></extra>"
))

# vertical line for average
fig.add_vline(
    x=3,  # this will be placed around the 11 to <12 hours band
    line_width=3,
    line_dash="dash",
    annotation_text=f"Average inactivity = {avg_inactivity} hours",
    annotation_position="top"
)

fig.update_layout(
    title="Daily Inactivity Distribution for Ages 18–24",
    xaxis_title="Inactivity duration",
    yaxis_title="Percentage",
    template="plotly_white",
    height=600,
    title_x=0.5
)

fig.show()

In [26]:

# Prepare data for line chart with averages
main_cleaned = cleaned[
    cleaned["metric_group"].isin(["inactivity", "moderate_vigorous"])
].copy()

main_cleaned["metric_group"] = main_cleaned["metric_group"].replace({
    "inactivity": "Daily Inactivity",
    "moderate_vigorous": "Moderate + Vigorous Activity"
})

# Calculate average for each metric group
averages = main_cleaned.groupby("metric_group")["value"].mean().reset_index()
averages.columns = ["metric_group", "average"]

print("Averages for each category:")
print(averages)

# Create line chart
fig = go.Figure()

for metric in main_cleaned["metric_group"].unique():
    metric_data = main_cleaned[main_cleaned["metric_group"] == metric]
    
    fig.add_trace(go.Scatter(
        x=metric_data["category"],
        y=metric_data["value"],
        mode="lines+markers",
        name=metric,
        line=dict(width=3),
        marker=dict(size=8),
        hovertemplate="Duration: %{x}<br>Percentage: %{y:.1f}%<extra></extra>"
    ))

# Add average lines
for _, row in averages.iterrows():
    fig.add_hline(
        y=row["average"],
        line_dash="dash",
        annotation_text=f"{row['metric_group']} Avg: {row['average']:.2f}%",
        annotation_position="right"
    )

fig.update_layout(
    title="Movement Profile for Ages 18–24",
    xaxis_title="Duration band",
    yaxis_title="Percentage",
    template="plotly_white",
    height=600,
    title_x=0.5,
    hovermode="x unified",
    xaxis_tickangle=-30
)

fig.show()

Averages for each category:
                   metric_group    average
0              Daily Inactivity  12.512500
1  Moderate + Vigorous Activity  14.285714


## Step 8: Identify the most common duration band in each metric group

A simple way to summarise each distribution is to find the **modal band**, which is the category with the highest percentage.

This gives one practical takeaway for each metric without forcing unlike categories into the same chart.


In [27]:
top_bands = (
    cleaned.sort_values(["metric_group", "value"], ascending=[True, False])
    .groupby(["metric_group", "metric_label"], as_index=False)
    .first()[["metric_group", "metric_label", "category", "value"]]
    .rename(columns={"category": "top_band", "value": "top_band_percentage"})
)

top_bands["top_band_percentage"] = top_bands["top_band_percentage"].round(1)

print(top_bands)


        metric_group                  metric_label                  top_band  \
0         inactivity              Daily inactivity  11 to less than 12 hours   
1     light_activity       Light physical activity    2 to less than 3 hours   
2           moderate    Moderate physical activity  2 to less than 2.5 hours   
3  moderate_vigorous  Moderate + vigorous activity  2 to less than 2.5 hours   
4           vigorous    Vigorous physical activity               1–4 minutes   

   top_band_percentage  
0                 29.7  
1                 43.7  
2                 28.4  
3                 25.9  
4                 59.6  


## Step 9: Estimate midpoint durations for a weighted-average summary

The source data is grouped into duration bands rather than individual observations.  
To estimate an average duration, each category is converted into an approximate midpoint.

### Important note
These midpoint values are estimates, especially for open-ended categories such as:
- `15 hours or more`
- `10 minutes or more`

So the weighted averages below should be interpreted as **approximate summaries**, not exact measured means.


In [28]:
def estimate_duration_hours(label: str) -> float:
    text = str(label).strip().lower()
    text = re.sub(r"(?<=\d)[^\w\s\.](?=\d)", "-", text)

    if text == "0 minutes":
        return 0.0

    match = re.fullmatch(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s+minutes", text)
    if match:
        low = float(match.group(1))
        high = float(match.group(2))
        return ((low + high) / 2) / 60

    match = re.fullmatch(r"less than\s+(\d+(?:\.\d+)?)\s+minutes", text)
    if match:
        high = float(match.group(1))
        return (high / 2) / 60

    match = re.fullmatch(r"(\d+(?:\.\d+)?)\s+minutes\s+to\s+less\s+than\s+(\d+(?:\.\d+)?)\s+hour", text)
    if match:
        low = float(match.group(1)) / 60
        high = float(match.group(2))
        return (low + high) / 2

    match = re.fullmatch(r"(\d+(?:\.\d+)?)\s+minutes\s+or\s+more", text)
    if match:
        low = float(match.group(1)) / 60
        return low + (5 / 60)

    match = re.fullmatch(r"less than\s+(\d+(?:\.\d+)?)\s+hours", text)
    if match:
        high = float(match.group(1))
        return high - 0.5

    match = re.fullmatch(r"(\d+(?:\.\d+)?)\s+to\s+less\s+than\s+(\d+(?:\.\d+)?)\s+hours", text)
    if match:
        low = float(match.group(1))
        high = float(match.group(2))
        return (low + high) / 2

    match = re.fullmatch(r"(\d+(?:\.\d+)?)\s+hours\s+or\s+more", text)
    if match:
        low = float(match.group(1))
        return low + 0.5

    return None


cleaned["midpoint_hours"] = cleaned["category"].astype(str).apply(estimate_duration_hours)
cleaned["midpoint_hours"] = pd.to_numeric(cleaned["midpoint_hours"], errors="coerce")

print(cleaned[["metric_group", "category", "midpoint_hours"]])


         metric_group                        category  midpoint_hours
0          inactivity               Less than 9 hours        8.500000
1          inactivity         9 to less than 10 hours        9.500000
2          inactivity        10 to less than 11 hours       10.500000
3          inactivity        11 to less than 12 hours       11.500000
4          inactivity        12 to less than 13 hours       12.500000
5          inactivity        13 to less than 14 hours       13.500000
6          inactivity        14 to less than 15 hours       14.500000
7          inactivity                15 hours or more       15.500000
8      light_activity               Less than 2 hours        1.500000
9      light_activity          2 to less than 3 hours        2.500000
10     light_activity          3 to less than 4 hours        3.500000
11     light_activity                 4 hours or more        4.500000
12           moderate            Less than 30 minutes        0.250000
13           moderat

## Step 10: Calculate approximate weighted averages

The weighted average is calculated as:

Estimated average duration =  
sum of (`midpoint_hours * percentage`) divided by sum of percentages

This provides one approximate average duration for each metric group.


In [29]:
weighted_summary = cleaned.copy()
weighted_summary["weighted_hours"] = weighted_summary["midpoint_hours"] * weighted_summary["value"]

avg_duration = (
    weighted_summary.groupby(["metric_group", "metric_label"], as_index=False)
    .agg(
        avg_hours=("weighted_hours", lambda s: s.sum() / weighted_summary.loc[s.index, "value"].sum())
    )
)

avg_duration["avg_minutes"] = (avg_duration["avg_hours"] * 60).round(1)
avg_duration["avg_hours"] = avg_duration["avg_hours"].round(2)

print(avg_duration)


        metric_group                  metric_label  avg_hours  avg_minutes
0         inactivity              Daily inactivity      11.64        698.6
1     light_activity       Light physical activity       2.58        154.7
2           moderate    Moderate physical activity       1.97        118.0
3  moderate_vigorous  Moderate + vigorous activity       2.05        123.2
4           vigorous    Vigorous physical activity       0.08          5.0


## Step 11: Key findings

From the cleaned and checked data:

- the distributions now sit inside the correct metric groups
- each group can be checked independently to confirm the percentages are complete
- separate charts are more appropriate than one combined chart because the duration bands use different units and ranges
- the modal-band table gives a quick summary of where the largest share of 18-24-year-olds falls for each metric
- the weighted averages provide an approximate duration summary for each activity pattern

This version of the notebook is designed to be easier to explain, easier to validate, and less likely to produce misleading visuals.
